# top_5_lsv_items — facts JSON shape

Runs `OSA_STORE_ACTIONABLE` detection plus the Stage 3 facts gatherers against live
Snowflake data, then prints the shape of each signal's `facts` dict — with
`top_5_lsv_items` (the renamed `top_lsv_items_last_week_top5`) dumped in full.

Tweak `CLIENT`, `QC_STORE_IDS`, and `ANALYSIS_DATE_STR` and re-run.
Keep `QC_STORE_IDS` small: the vs_chain baseline is computed over this subset only.

In [ ]:
# ---- Configuration ----------------------------------------------------------------
CLIENT = 'eastofengland'  # client subfolder under config/clients/
ANALYSIS_DATE_STR = '2026-07-26'
QC_STORE_IDS = [451, 497, 463, 416, 479]  # small subset — chain baseline = these stores
ENABLED_DETECTORS = ['osa_store_actionable.wow', 'osa_store_actionable.vs_chain']

import datetime
analysis_date = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks

print(f'Client        : {CLIENT}')
print(f'This week     : {THIS_WEEK_START} -> {THIS_WEEK_END}')
print(f'Last week     : {LAST_WEEK_START} -> {LAST_WEEK_END}')
print(f'Store filter  : {QC_STORE_IDS}')

In [ ]:
# ---- Setup: path, config, session -------------------------------------------------
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg = load_config(CLIENT)
cfg.detectors = cfg.detectors.model_copy(update={'enabled': ENABLED_DETECTORS})

session = create_session(cfg)
print(f'Enabled detectors: {cfg.detectors.enabled}')

In [ ]:
# ---- Load OSA + hierarchies --------------------------------------------------------
from snowflake.snowpark import functions as F

osa_raw = (
    session.table(cfg.tables.osa)
    .filter(F.col('DATE') >= LOOKBACK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .filter(F.col('STORE_ID').isin(QC_STORE_IDS))
    .select(
        F.col('STORE_ID'),
        F.col('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('DATE'),
        F.col('INVENTORY_QTY'),
        F.col('DEPARTMENT_ID'),
        F.col('REPORTING_PERIOD_ID'),
    )
)

item = (
    session.table(cfg.tables.hierarchy)
    .select(
        F.col('ITEM_CODE'),
        F.col('ITEM_DESCRIPTION'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('ITEM_HIERARCHY_2').is_not_null())
    .cache_result()
)

# Used by the item_lsv_changes gatherer for UPC/EAN arrays.
wmd_item_hierarchy = (
    session.table(cfg.tables.wmd_hierarchy)
    .select(
        F.col('UPC'),
        F.col('OSA_ITEM_CODE'),
        F.col('ITEM_DESCRIPTION'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
    .cache_result()
)

print(f'OSA rows (QC stores) : {osa.count():,}')
print(f'WMD hierarchy rows   : {wmd_item_hierarchy.count():,}')

In [ ]:
# ---- Run detectors (softened thresholds so alerts fire) -----------------------------
import dataclasses
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig

det_cfg = DetectorConfig.from_config(cfg.detector_config.model_dump() if cfg.detector_config else {})
det_cfg = dataclasses.replace(det_cfg, **{
    'min_actionable_wow_drop': 0.01,
    'min_actionable_chain_gap': 0.01,
    'min_actionable_lsv': 100.0,
    'min_actionable_positive_rows': 5,
    'max_actionable_unknown_row_pct': 50.0,
})

signals = detect_signals(
    osa,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    store_id=QC_STORE_IDS,
    item_hierarchy=wmd_item_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

actionable = [s for s in signals if s['signal_type'] == 'OSA_STORE_ACTIONABLE']
print(f'\nOSA_STORE_ACTIONABLE signals: {len(actionable)}')
for s in actionable[:10]:
    print(f"  store={s['store_id']}  dept={s.get('department')}  triggers={s.get('trigger_types')}")

In [ ]:
# ---- Run the facts stage ------------------------------------------------------------
from store_insight.pipeline.stages._3_facts.facts import enrich_signals

enriched = enrich_signals(
    actionable,
    osa,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    section='ITEM_HIERARCHY_2',
    item_hierarchy=wmd_item_hierarchy,
    cfg=cfg,
)
print(f'{len(enriched)} signals enriched')

In [ ]:
# ---- Shape of the facts dict ---------------------------------------------------------
# Scalars -> type name, lists -> one-element skeleton + item count.
import json


def skeleton(value):
    if isinstance(value, dict):
        return {k: skeleton(v) for k, v in value.items()}
    if isinstance(value, list):
        if not value:
            return []
        first = skeleton(value[0])
        return [first] if len(value) == 1 else [first, f'... {len(value)} items']
    return type(value).__name__


sig = enriched[0]
print(f"signal: store={sig['store_id']}  dept={sig.get('department')}  triggers={sig.get('trigger_types')}\n")
print(json.dumps(skeleton(sig['facts']), indent=2))

In [ ]:
# ---- top_5_lsv_items for the first few signals ----------------------------------------
for sig in enriched[:3]:
    print(f"store={sig['store_id']}  dept={sig.get('department')}")
    print(json.dumps(sig['facts'].get('top_5_lsv_items', []), indent=2))
    print()

In [ ]:
# ---- Full facts JSON for one signal ----------------------------------------------------
print(json.dumps(enriched[0]['facts'], indent=2, default=str))